## Dynamic Temperature
Modify the temperature during generation. We start with a higher temperature to enforce diversity but then we decrease it after each generation to make the output more stable and roboust. It makes sense since the start of the generation also affects the subsequent tokens. I made this linear but we could also try exponential or whatever.


Sampling Parameters:
```python
use_dynamic_temperature: bool = False
initial_temperature: Optional[float] = None   # starting temperature
final_temperature: Optional[float] = None     # final temperature
max_steps: Optional[torch.Tensor] = None      # maximum number of steps
```


Metadata:
```python
current_step: Optional[torch.Tensor] = None   # current generation step
```

We keep track of the current step via the metadata (so we update the object in the model runner). That info was probably present somewhere else in vllm but whatever, that doesn't add any overhead (it's just a matter of increasing a value in a variable)

## XTC (Exclude Top Choice)
Basically exclude N (parameter) top choices, given also a probability threshold and only for choices with a minimum requested probability.

Sampling Parameters:
```python
use_xtc: bool = False
xtc_exclude_top: Optional[int] = None   # how many to exclude
xtc_exclusion_threshold: Optional[float] = None   # what is the probability threshold
xtc_min_probability: Optional[float] = None     # what is the minimum probability to consider for excluding
```

An important note is that I implemented this in a sub-optimal way. THe best way to do this would be to modify the top-p top-k sampler such that when we use XTC that method doesn't compute the softmax and works only in logits space (therefore also removing the `random_sample` method from the inner forward of the sampler class). This is due to the fact that now, in order to use the parameters for XTC we have to go in probability space and then go back to logits space, which is very very wasteful (and I think this is the main reason why XTC is the slowest method in the benchmark). However that structural change was a bit too long to persue and I'm not going to fix vllm's poor design choices for a coding challange. We can eventually improve this when I get hired :D

so, tldr is XTC is slow because we do logits -> probs -> logits and then after again we go to probs. 